# Создание цикла обучения

Data loader 

In [10]:
import os
import torch
import pandas as pd
from torchvision.io import decode_image
from torch.utils.data import Dataset
import torch.nn.functional as F
from torchvision.transforms import v2


1. Создание кастомного класса для загрузки данных

In [39]:
class MNISTDataset(Dataset):
    def __init__(self, annotations_file, transforms = v2.ToDtype(torch.float32, scale=True) , target_transform = F.one_hot):
        df = pd.read_csv(annotations_file)
        self.img_labels , self.image = df['label'], df.drop(columns='label')
        self.target_transform = target_transform
        self.transform = transforms

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        label = self.img_labels.iloc[idx]
        image = torch.tensor(self.image.iloc[idx].values, dtype= torch.uint8 )
        image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(torch.tensor(label),num_classes=10).float()
        return image, label

2. Деление на тестовыи и треин данные 

In [40]:
from torch.utils.data import random_split , DataLoader
data = MNISTDataset('fashion-mnist_test.csv')
train_size = int(0.7 * len(data))
val_size = int(0.15 * len(data))
test_size = len(data) - train_size - val_size

train_data, val_data, test_data = random_split(
    data, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_data, batch_size = 64, shuffle = True)
val_loader =  DataLoader(val_data, batch_size = 64, shuffle = False)
test_loader =  DataLoader(test_data, batch_size = 64, shuffle = False)


In [41]:
for X, y in test_loader:
    print(f"Shape of X [N, C, H, W]: {X.shape},  {X.dtype}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 784]),  torch.float32
Shape of y: torch.Size([64, 10]) torch.float32


то есть всё хорошо поделилось на батчи и данные уже преведены в формат 2D с которым принято работать для данного датасета , а вектор ответов преобразован в 10 векторов с значением 0 или 1 при помощи one hot encoding 

3. Создание модели

In [42]:
from torch import nn
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Linear(256, 10))
        
    def forward(self,x):
        logits = self.linear_relu(x)
        return logits
    

model = MNISTModel()
print(model)

MNISTModel(
  (linear_relu): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


Let's check model's correct

In [43]:
input = torch.rand([16,784], dtype= torch.float32)
print((model(input)).shape)

torch.Size([16, 10])


Chosing loss function and optimisator 

In [50]:
from torch.optim import Adam 

loss_cross_entr = nn.CrossEntropyLoss()
opt = Adam(model.parameters(), lr = 0.001)

In [37]:
print(train_loader)

Model traning

In [ ]:
from torch.autograd import backward

EPOCHS = 4 
for epoch in range(EPOCHS):
    train_loss = []
    test_loss = []

    model.train()
    for x , target in train_loader:
        running_train_loss = []
        pred = model(x)
        loss = loss_cross_entr(pred, target)

        running_train_loss.append(loss.item())
        mean_tr_loss = sum(running_train_loss)/len(running_train_loss)

        opt.zero_grad()
        loss.backward()
        opt.step()

    train_loss.append(mean_tr_loss)

    model.eval()
    for x , target in test_loader:
        running_test_loss = []
        pred = model(x)
        loss = loss_cross_entr(pred, target)
        running_test_loss.append(loss.item())
        mean_tr_loss = sum(running_test_loss)/len(running_test_loss)

    test_loss.append(mean_tr_loss)

    print(f'{epoch+1}/{EPOCHS}, train loss = {train_loss}, test loss = {test_loss}')

1/4, train loss = [0.44347384572029114], test loss = [0.1974353790283203]
2/4, train loss = [0.16317534446716309], test loss = [0.2880076467990875]
